# PDU breach-risk batch scoring

Scores every PDU with the `champion` breach-risk model at the latest 00:00 / 12:00 UTC scoring time and writes the result to `pdu_breach_risk_predictions` (one row per unit per scoring time, idempotent per scoring time). Features are computed point-in-time by the same Feature Views definitions used in training (`src/ml/pdu_pm_features.py`). Runs as the 12-hourly scoring job; the monitoring app reads the latest batch.

In [0]:
%pip install -q "databricks-feature-engineering>=0.16.0" lightgbm
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "mdk_demo_catalog")
dbutils.widgets.text("schema", "pdu_monitoring")
dbutils.widgets.text("ops_history_schema", "ops_history")
dbutils.widgets.text("maintenance_table", "maintenance_actions")
dbutils.widgets.text("model_name", "pdu_breach_risk_model")
dbutils.widgets.text("model_alias", "champion")
dbutils.widgets.text("predictions_table", "pdu_breach_risk_predictions")
dbutils.widgets.text("scoring_ts", "")       # empty = latest 00:00 / 12:00 UTC boundary
dbutils.widgets.text("top_k", "10")           # technician capacity per 12 h shift
dbutils.widgets.text("elevated_k", "25")

import os, sys, importlib
ML_DIR = os.path.abspath("../ml")
if ML_DIR not in sys.path:
    sys.path.append(ML_DIR)
import pdu_pm_features as pm
pm = importlib.reload(pm)

cfg = pm.PMConfig(catalog=dbutils.widgets.get("catalog"), schema=dbutils.widgets.get("schema"),
                  ops_schema=dbutils.widgets.get("ops_history_schema"),
                  maintenance_table=dbutils.widgets.get("maintenance_table"))
MODEL_NAME = f"{cfg.catalog}.{cfg.schema}.{dbutils.widgets.get('model_name')}"
ALIAS = dbutils.widgets.get("model_alias")
PRED_TABLE = f"{cfg.catalog}.{cfg.schema}.{dbutils.widgets.get('predictions_table')}"
TOP_K, ELEVATED_K = int(dbutils.widgets.get("top_k")), int(dbutils.widgets.get("elevated_k"))
SCORING_TS = dbutils.widgets.get("scoring_ts") or pm.latest_scoring_ts(spark, cfg)
print(MODEL_NAME, ALIAS, PRED_TABLE, SCORING_TS, sep="\n")

In [0]:
import mlflow
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
mv = MlflowClient(registry_uri="databricks-uc").get_model_version_by_alias(MODEL_NAME, ALIAS)
model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@{ALIAS}")
meta = mlflow.artifacts.load_dict(f"runs:/{mv.run_id}/feature_columns.json")
FEATURE_LIST, MODEL_COLS, RAW = meta["feature_list"], meta["columns"], meta["raw_features"]
INCLUDE_MAINT = any("repair" in n for n in RAW)
print(f"{MODEL_NAME} v{mv.version} ({FEATURE_LIST}): {len(MODEL_COLS)} model columns, {len(RAW)} Feature Views features")

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()
features = pm.get_or_create_features(fe, cfg, RAW, include_maintenance=INCLUDE_MAINT)
frame = pm.scoring_frame(spark, cfg, SCORING_TS)
pdf = pm.add_derived_features(
    fe.create_training_set(df=frame, features=features, label=None).load_df().toPandas())

# Units already in a sustained breach just before scoring time are a monitoring problem, not a
# prediction: flag them and rank only the others.
flags = (pm.label_spine(spark, cfg, start_ts=str(SCORING_TS), end_ts=str(SCORING_TS))
         .where(F.col("scoring_ts") == F.lit(SCORING_TS).cast("timestamp"))
         .select("unit_id", "already_breaching").toPandas())
pdf = pdf.merge(flags, on="unit_id", how="left")
pdf["already_breaching"] = pdf["already_breaching"].fillna(False).astype(bool)

pdf["risk_score"] = model.predict_proba(pdf[MODEL_COLS])[:, 1]
ranked = pdf.loc[~pdf.already_breaching].sort_values("risk_score", ascending=False)
pdf["risk_rank"] = pd.Series(range(1, len(ranked) + 1), index=ranked.index)
pdf["risk_band"] = "LOW"
pdf.loc[pdf.risk_rank <= ELEVATED_K, "risk_band"] = "ELEVATED"
pdf.loc[pdf.risk_rank <= TOP_K, "risk_band"] = "HIGH"
pdf.loc[pdf.already_breaching, "risk_band"] = "IN_BREACH"
print(pdf["risk_band"].value_counts().to_string())

In [0]:
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {PRED_TABLE} (
    unit_id STRING NOT NULL, scoring_ts TIMESTAMP NOT NULL,
    risk_score DOUBLE, risk_rank INT, risk_band STRING, already_breaching BOOLEAN,
    rack_id STRING, unit_model STRING, shelf INT, position INT,
    temp_avg_1h DOUBLE, temp_max_6h DOUBLE, temp_headroom_1h DOUBLE, temp_trend_1h_vs_6h DOUBLE,
    temp_trend_6h_vs_24h DOUBLE, current_avg_1h DOUBLE, current_std_6h DOUBLE, volt_min_6h DOUBLE,
    model_name STRING, model_version STRING, feature_list STRING, scored_at TIMESTAMP,
    CONSTRAINT pdu_breach_risk_predictions_pk PRIMARY KEY (unit_id, scoring_ts))
  COMMENT 'Breach-risk predictions per PDU per 12 h scoring time: risk_score = model score for a sustained breach (>= 5 min outside breach limits) in the next 12 h. Use risk_rank / risk_band for prioritisation; scores are not calibrated probabilities.'
  TBLPROPERTIES (delta.enableChangeDataFeed = true)""")

keep = ["unit_id", "scoring_ts", "risk_score", "risk_rank", "risk_band", "already_breaching",
        "rack_id", "unit_model", "shelf", "position", "temp_avg_1h", "temp_max_6h", "temp_headroom_1h",
        "temp_trend_1h_vs_6h", "temp_trend_6h_vs_24h", "current_avg_1h", "current_std_6h", "volt_min_6h"]
out = (spark.createDataFrame(pdf[keep].astype({"risk_rank": "Int64"}))
       .withColumn("scoring_ts", F.col("scoring_ts").cast("timestamp"))
       .withColumn("risk_rank", F.col("risk_rank").cast("int"))
       .withColumn("shelf", F.col("shelf").cast("int")).withColumn("position", F.col("position").cast("int"))
       .withColumn("model_name", F.lit(MODEL_NAME)).withColumn("model_version", F.lit(str(mv.version)))
       .withColumn("feature_list", F.lit(FEATURE_LIST)).withColumn("scored_at", F.current_timestamp()))
ts_sql = pd.Timestamp(SCORING_TS).strftime("%Y-%m-%d %H:%M:%S")
(out.write.mode("overwrite").option("replaceWhere", f"scoring_ts = timestamp'{ts_sql}'")
    .saveAsTable(PRED_TABLE))
display(spark.table(PRED_TABLE).where(f"scoring_ts = timestamp'{ts_sql}'").orderBy("risk_rank"))